# Data Source Extraction

This notebook explains the output shape for data-source extraction, shows
lightweight checks for reviewing extracted items, and then defines a **new
miner declaratively** with a `TaskSpec` — the extraction counterpart to the
declarative classifier in `07_classification_workflow.ipynb`.

## Review The Task Configuration

`FindDataSourcesConfig` defines the retrieval templates and prompt used by `PrecisionMiner` for data-source extraction.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Locate `notebooks/episcope_nb.py`, the shared helper module. This works whether
# the kernel starts in `notebooks/` or at the repository root.
_cwd = Path.cwd()
_nb_dir = next(
    (
        directory
        for candidate in [_cwd, *_cwd.parents]
        for directory in (candidate, candidate / "notebooks")
        if (directory / "episcope_nb.py").is_file()
    ),
    None,
)
if _nb_dir is None:
    raise FileNotFoundError("Could not find notebooks/episcope_nb.py")
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

# Importing the helper also puts `src/` on sys.path when this is a checkout.
import episcope_nb as nb

WORK_DIR = nb.bootstrap()
WORK_DIR

In [ ]:
from episcope.workflows.precision_miner import FindDataSourcesConfig

config = FindDataSourcesConfig(top_k=5)
print("top_k:", config.top_k)
print("section_filters:", config.section_filters)
print("structured_output:", config.structured_output)
print("retrieval templates:")
for template in config.retrieval_templates[:4]:
    print("-", template)

`structured_output` controls decode-time enforcement and defaults to `"schema"`.
When the workflow runs, it passes `response_schema=ExtractionResultSchema` down to
the generator, so a provider that supports native structured output is *constrained*
to emit conforming JSON rather than merely asked to. `"json"` requests valid JSON
only, and `"off"` applies no constraint. The demo generators below ignore the
kwarg (they return fixed JSON), but a real `LLMGenerator` will use it.

## Validate Generator JSON

The parser accepts a single JSON object with a task-level description and an `items` list. Each item should identify the source, explain why it matters, and preserve the raw supporting text when possible.


In [ ]:
import json

from episcope.workflows.precision_miner.parsing import PrecisionMinerResponseParser

raw_response = json.dumps(
    {
        "description": "The study uses registry data and a public repository.",
        "items": [
            {
                "name": "National Hospital Registry",
                "url": None,
                "explanation": "Named as the source of patient admissions and outcomes.",
                "raw_text": "The analysis used patient records from the National Hospital Registry.",
            },
            {
                "name": "Public analysis repository",
                "url": "https://example.org/repository",
                "explanation": "Hosts de-identified trial data and code.",
                "raw_text": "De-identified trial data and the analysis code are available from the public repository.",
            },
        ],
    }
)

parsed = PrecisionMinerResponseParser.parse(raw_response)
parsed.model_dump()


## Review Evidence Coverage

A useful extraction should be tied back to retrieved chunks. The review table below is a simple pattern for checking whether each item has enough support.


In [ ]:
from episcope.schemas import SearchResult

candidate_evidence = [
    SearchResult(
        id="chunk-1",
        paper_id="paper_open_data",
        section_title="Methods",
        section_type="Methods",
        text="The analysis used patient records from the National Hospital Registry.",
        similarity_score=0.91,
    ),
    SearchResult(
        id="chunk-2",
        paper_id="paper_open_data",
        section_title="Data availability",
        section_type="Data availability",
        text="De-identified trial data and the analysis code are available from the public repository.",
        similarity_score=0.86,
    ),
]

review_rows = []
for item in parsed.items:
    supporting = [chunk for chunk in candidate_evidence if item.raw_text and item.raw_text[:40] in chunk.text]
    review_rows.append(
        {
            "name": item.name,
            "has_url": bool(item.url),
            "supporting_chunks": len(supporting),
            "best_section": supporting[0].section_title if supporting else None,
        }
    )

review_rows

## Define A New Miner Declaratively

`FindDataSourcesConfig` is a built-in. To extract something else, you do not need
a new Python class: a *declarative task spec* with `kind: "miner"` is enough.
A miner spec is simpler than a classifier spec — there are no labels, just the
`retrieval_templates` that gather evidence, plus optional `section_filters` and
prompt overrides.

`build_miner_config_from_spec` turns the spec into a runnable
`PrecisionMinerConfig`; the prompt and the JSON schema are supplied for you.

In [ ]:
from episcope.workflows.registry import TaskSpec, build_miner_config_from_spec

funding_spec = TaskSpec.model_validate(
    {
        "key": "find_funding_sources",
        "kind": "miner",
        "label": "Funding sources",
        "description": "Grants, agencies, and sponsors that funded the study.",
        "top_k": 4,
        "retrieval_templates": [
            "Who funded this study? Which grants or awards supported the work?",
            "Which agencies, foundations, or sponsors are acknowledged?",
            "What are the grant numbers or award identifiers?",
        ],
        # Restrict evidence to sections where funding is usually declared.
        # Use None (the default) to search the whole paper.
        "section_filters": None,
    }
)

funding_config = build_miner_config_from_spec(funding_spec)
print("top_k:", funding_config.top_k)
print("structured_output:", funding_config.structured_output)
print("templates:", len(funding_config.retrieval_templates))
print("\nGenerated system prompt:\n", funding_config.system_prompt)

## Run The Miner On Fixed Evidence

The other notebooks build a real index. Here the point is the *contract*, not
retrieval quality, so `nb.StaticRetriever` returns a fixed evidence list. That is
enough because `PrecisionMiner` only ever calls `retrieve_by_paper` on the
retriever — the type hint says `BaseRetriever`, but nothing else is required.
Passing `metadata=` to `run_detailed` likewise avoids needing an `AcademicDB`.

This is a useful pattern for testing a new miner spec: pin the evidence, and any
change in the output comes from the prompt or the model, not from retrieval drift.

In [ ]:
from typing import Any, Callable, Optional, Sequence

from episcope.rag.generation.base import Generator
from episcope.rag.provenance import Evidence, Provenance
from episcope.schemas import PaperMetadata
from episcope.workflows import PrecisionMiner

funding_evidence = [
    SearchResult(
        id="chunk-3",
        paper_id="paper_open_data",
        section_title="Funding",
        section_type="Acknowledgements",
        text=(
            "This work was supported by the National Institute for Health Research "
            "under grant NIHR-2021-4417."
        ),
        similarity_score=0.88,
    ),
    SearchResult(
        id="chunk-4",
        paper_id="paper_open_data",
        section_title="Acknowledgements",
        section_type="Acknowledgements",
        text=(
            "Additional support was provided by the Open Data Foundation. "
            "The funders had no role in study design or analysis."
        ),
        similarity_score=0.74,
    ),
]


class DemoFundingGenerator(Generator):
    """Stands in for `LLMGenerator`; returns JSON matching the extraction schema.

    Unlike the helper's `nb.FixedJSONGenerator`, this one reads the retrieved
    contexts, so `raw_text` traces back to the evidence the miner actually saw.
    """

    model_id = "demo-funding-generator"

    def generate(
        self,
        contexts: Sequence[Any],
        *,
        question: Optional[str] = None,
        message_builder: Optional[Callable[..., Any]] = None,
        **kwargs: Any,
    ) -> Provenance:
        contexts = list(contexts)
        payload = {
            "description": "The study was funded by one public agency and one foundation.",
            "items": [
                {
                    "name": "National Institute for Health Research (NIHR-2021-4417)",
                    "url": None,
                    "explanation": "Named as the primary funder, with a grant number.",
                    "raw_text": contexts[0].text if contexts else "",
                },
                {
                    "name": "Open Data Foundation",
                    "url": None,
                    "explanation": "Acknowledged as providing additional support.",
                    "raw_text": contexts[1].text if len(contexts) > 1 else "",
                },
            ],
        }
        evidences = [
            Evidence(
                paper_id=chunk.paper_id,
                snippet=chunk.text,
                section=chunk.section_type,
                model_id=self.model_id,
                prompt_id="demo-funding",
            )
            for chunk in contexts
        ]
        return Provenance(answer=json.dumps(payload), evidences=evidences)


funding_miner = PrecisionMiner(
    retriever=nb.StaticRetriever(funding_evidence),
    generator=DemoFundingGenerator(),
    config=funding_config,
)

detailed = funding_miner.run_detailed(
    "paper_open_data",
    metadata=PaperMetadata(
        title="Trial data sharing and reuse",
        abstract="A randomized trial reused a public patient-level dataset.",
    ),
)

for item in detailed.result.items:
    print(f"- {item.name}\n    {item.explanation}")

## The Same Miner Without Python

A miner spec is just data, so the notebook version above and a JSON file are
interchangeable:

- **Scaffold + validate**: `episcope tasks new --kind miner > funding.json`, then
  `episcope tasks validate --task-file funding.json`. Add `--interactive` to
  `tasks new` to answer prompts instead of editing the scaffold by hand.
- **Run from the CLI**: `episcope precision-miner --file paper.pdf --task-file funding.json`,
  or drop the file in `<workspace>/tasks/` and select it by key with `--miner-kind`.
- **Register in-process**: `register_task(funding_spec)` makes the key visible to
  `miner_catalog()`, the API's `/health` catalog, and the Streamlit dropdowns.

See the [Declarative Tasks](https://github.com/VinsRR/EpiScope/wiki/Declarative-Tasks)
wiki page for the full spec format.

## Summary

Keep the compact `ExtractionResult` for downstream tables and dashboards. Keep the
detailed workflow trace during review so uncertain items can be traced back to
prompt messages, raw generator output, and supporting chunks.

For a new extraction task, reach for a declarative miner `TaskSpec` first — you
only supply `retrieval_templates` and optional `section_filters`. Write a
`PrecisionMinerConfig` by hand only when you need a bespoke prompt or a schema the
generated one cannot express. The classifier equivalent of this choice is covered
in `07_classification_workflow.ipynb`.